# Notebook 2: Train Continual SD-LoRA Adapter

Bu notebook tek crop icin adapter egitir, OOD hazirligini olcer ve export ciktisini kaydeder.

Onerilen kullanim sirasi:
1. Calisma kimligi hucresinde `CROP_NAME` ve `PART_NAME` belirleyin.
2. Parametreleri duzenleyin.
3. Guncelleme/erisim kontrolu hucresini calistirin.
4. Hucreleri sirayla calistirin.
5. Is bitince `guided/00_start_here.md` ile ciktilari okuyun.


## Public quick start

This notebook runs training, OOD calibration, final evaluation and export for one selected adapter.

- The public default creates a small deterministic synthetic dataset, so no private dataset token is needed.
- The sample is only for exercising the pipeline; it is never production or model-quality evidence.
- Change `ADAPTER_KEY` if desired, add a gated-model Hugging Face token, then run the cells in order.
- Maintainers can opt into the private research dataset with `DATASET_SOURCE_KIND = "github_release"`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def _configure_colab_git_read_access():
    token = str(os.environ.get('AADS_GITHUB_RELEASE_READ_TOKEN', '')).strip()
    if not token:
        try:
            from google.colab import userdata
            token = str(userdata.get('AADS_GITHUB_RELEASE_READ_TOKEN') or '').strip()
        except Exception:  # Colab secret access raises provider-specific exceptions.
            token = ''
    os.environ['GIT_TERMINAL_PROMPT'] = '0'
    if not token:
        return
    askpass = Path('/tmp/aads_git_read_askpass.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "*Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "*) printf '%s\\n' \"$AADS_GIT_READ_TOKEN\" ;;\n"
        'esac\n',
        encoding='utf-8',
    )
    askpass.chmod(0o700)
    os.environ['GIT_ASKPASS'] = str(askpass)
    os.environ['GIT_ASKPASS_REQUIRE'] = 'force'
    os.environ['AADS_GIT_READ_TOKEN'] = token
    os.environ['AADS_GITHUB_RELEASE_READ_TOKEN'] = token


_configure_colab_git_read_access()

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/aads-open-world-plant-disease.git')
NOTEBOOK2_SPARSE_PATHS = (
    'README.md',
    'docs',
    'src',
    'scripts',
    'config',
    'colab_notebooks',
    'requirements.txt',
    'requirements_colab.txt',
    'pyproject.toml',
)



def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', REPO_URL, str(CLONE_TARGET)], check=True)
        subprocess.run(['git', 'sparse-checkout', 'set', *NOTEBOOK2_SPARSE_PATHS], cwd=str(CLONE_TARGET), check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET


_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell03_runtime_setup.py', globals())


In [ ]:
# Public smoke profile: normally only change ADAPTER_KEY.
ADAPTER_KEY = "grape__fruit"  # grape__fruit, grape__leaf, strawberry__fruit, strawberry__leaf, tomato__fruit, tomato__leaf, apricot__fruit, apricot__leaf

DATASET_SOURCE_KIND = "public_sample"  # public_sample or github_release
PUBLIC_SAMPLE_ROOT = "data/public_sample_runtime_datasets"

# Private research dataset settings are used only when DATASET_SOURCE_KIND="github_release".
DATASET_RELEASE_REPOSITORY = "EfeErim/aads-open-world-plant-disease"
DATASET_RELEASE_TAG = "aads-dataset-v1.0.0"
DATASET_RELEASE_CACHE_ROOT = ".runtime_tmp/dataset_release_cache"

# Public smoke runs do not reuse private optimization history.
ENABLE_BAYESIAN_OPTIMIZATION = False

# Adapter bazli defaultlar tek repo helper'indan gelir; Notebook 6 da ayni kaynagi kullanir.
from scripts.notebook_helpers.adapter_recommendations import get_adapter_recs
ADAPTER_RECS = get_adapter_recs()

# Adapterden bagimsiz gorunur defaultlar.
DEFAULT_RUNTIME_PARAMS = {
    "ENABLE_BAYESIAN_OPTIMIZATION": ENABLE_BAYESIAN_OPTIMIZATION,
    "WEIGHT_DECAY": 0.01,
    "LOSS_NAME": "logitnorm",
    "LOGITNORM_TAU": 1.0,
    "MIXED_PRECISION": "bf16",
    "GRAD_ACCUM_STEPS": 1,
    "MAX_GRAD_NORM": 1.0,
    "LABEL_SMOOTHING": 0.0,
    "SCHEDULER_NAME": "cosine",
    "SCHEDULER_WARMUP_RATIO": 0.1,
    "SCHEDULER_MIN_LR": 1e-6,
    "EARLY_STOPPING_PATIENCE": 6,
    "EARLY_STOPPING_MIN_DELTA": 0.0,
    "DETERMINISTIC": False,
    "SEED": 42,
    "RANDAUGMENT_NUM_OPS": 2,
    "RANDAUGMENT_MAGNITUDE": 7,
    "NUM_WORKERS": 12,
    "PREFETCH": 8,
    "PIN_MEMORY": True,
    "USE_CACHE": True,
    "CACHE_TRAIN_SPLIT": True,
    "VALIDATION_EVERY_N_EPOCHS": 1,
    "CHECKPOINT_EVERY_N_STEPS": 250,
    "CHECKPOINT_ON_EXCEPTION": True,
    "STDOUT_BATCH_INTERVAL": 12,
    "RESUME_MODE": "fresh",
    "AUTO_DISCONNECT_RUNTIME": True,
    "AUTO_DISCONNECT_GRACE_SECONDS": 20,
    "AUTO_PUSH_TO_GITHUB": True,
    "AUTO_PUSH_REMOTE_NAME": "origin",
    "AUTO_PUSH_BRANCH": None,
}

# Sadece bu run icin defaultlari ezmek istersen buraya yaz.
MANUAL_PARAM_OVERRIDES = {}

with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    from scripts.notebook_helpers.cell_script_runner import run_cell_script
    run_cell_script('nb2_cell04_parameter_resolution.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell05_access_check.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell06_dataset_validation.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell07_engine_init.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell08_ood_config_verify.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell09_training.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell10_ood_calibration.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell11_adapter_save.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell12_final_evaluation.py', globals())
